#### 1. Obtain protein IDs of orthologs per GENE from NCBI
#### 2. Using the IDS Fetch protein sequence ID from 'records' from the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) 
#### 3. Store nested dictionary of protein : sequences
#### 4. identify and extract unique sequences, human first then non-human
#### 5. Save sequences in FASTA format for each gene in /Fasta folder


#### 

In [16]:
run Kondrashov

* function: fetch protein record of each ortholog from NCBI

In [17]:
def fetch_gene_record(gene_id):
    """
    Fetch one Entrez Gene XML record and parse it with xmltodict.
    """
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="xml", retmode="text")
    record = x2d.parse(handle.read().decode("utf-8"))
    handle.close()
    return record

* function: extract protein ID from dictionary into a 'list'

In [18]:
def extract_protein_accessions(obj):
    """
    Recursively search the NCBI Gene XML dictionary and collect protein accessions.
    Returns accessions like XP_077799762.1 or NP_000148.2.
    """
    proteins = set()

    if isinstance(obj, dict):
        acc = obj.get("Gene-commentary_accession")
        ver = obj.get("Gene-commentary_version")

        if acc is not None:
            # Protein accessions usually start with NP_, XP_, or YP_
            if acc.startswith(("NP_", "XP_", "YP_")):
                if ver is not None:
                    proteins.add(f"{acc}.{ver}")
                else:
                    proteins.add(acc)

        for value in obj.values():
            proteins.update(extract_protein_accessions(value))

    elif isinstance(obj, list):
        for item in obj:
            proteins.update(extract_protein_accessions(item))

    return proteins


## [Primates](https://meshb.nlm.nih.gov/record/ui?ui=D011323&dcmsLinks=true)
* RUN CODE USING DEFINED FUNCTIONS: obtain orthologs per gene 
* final output is 'records' contains gene, primate orthologs, details

In [87]:
run kondrashov

In [79]:
lociii
loci
for x in loci:
    if x not in lociii:
        print(f"'{x}',")

In [ ]:
xx = ['ABCD1',
'ALPL',
'AR',
'BTK',
'CASR',
'CFTR',
'CYBB',
'F7',
'F8',
'F9',
'G6PD',
'GJB1',
'HBB',
'HPRT1',
'IL2RG',
'KCNH2',
'L1CAM',
'MPZ',
'MYH7',
'PMM2',
'RHO',
'TP53',
'TTR']

In [88]:
lociii

['ABCA12',
 'ABCD1',
 'ACAD9',
 'ACADM',
 'ACADVL',
 'ACAT1',
 'ADA',
 'AGL',
 'AGXT',
 'AHI1',
 'AIRE',
 'ALMS1',
 'ALOX12B',
 'ALPL',
 'ANK1',
 'ANO5',
 'APC',
 'AR',
 'ARSA',
 'ARSB',
 'ASL',
 'ASPA',
 'ASPM',
 'ATM',
 'ATP7B',
 'BBS2',
 'BCKDHA',
 'BEST1',
 'BLM',
 'BRIP1',
 'BTD',
 'BTK',
 'CASR',
 'CBS',
 'CC2D2A',
 'CDH23',
 'CEP290',
 'CERKL',
 'CFTR',
 'CHD7',
 'CHEK2',
 'CLN3',
 'CNGB3',
 'CPS1',
 'CRB1',
 'CTNS',
 'CYBB',
 'CYP11B1',
 'CYP17A1',
 'CYP27A1',
 'DCLRE1C',
 'DICER1',
 'DUOX2',
 'EHMT1',
 'ELP1',
 'ERCC6',
 'EVC',
 'EVC2',
 'EXT1',
 'EXT2',
 'EYS',
 'F7',
 'F8',
 'F9',
 'FANCA',
 'FANCI',
 'FBN2',
 'FKRP',
 'FLCN',
 'FRAS1',
 'G6PD',
 'GAA',
 'GALC',
 'GALNS',
 'GALT',
 'GBA1',
 'GBE1',
 'GCDH',
 'GJB1',
 'GLB1',
 'GNE',
 'GNPTAB',
 'HBB',
 'HEXA',
 'HEXB',
 'HGD',
 'HGSNAT',
 'HMBS',
 'HPRT1',
 'HPS3',
 'IL2RG',
 'ITGA2B',
 'IVD',
 'KCNH2',
 'KCNQ1',
 'KRIT1',
 'L1CAM',
 'LDLR',
 'LIPA',
 'LOXHD1',
 'LPL',
 'LRPPRC',
 'LYST',
 'MAN2B1',
 'MCCC1',
 'MCCC2',
 'MED

In [36]:
# obtain primate Orthologs per gene
result = {}

for locus in lociii:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    search_record = Entrez.read(handle)
    handle.close()

    gene_ids = search_record["IdList"]
    print(f"{locus}: {search_record['Count']} -> {gene_ids[:4]}")


    result[locus] = {}

    for gene_id in gene_ids:
        try:
            gene_record = fetch_gene_record(gene_id)
            protein_ids = sorted(extract_protein_accessions(gene_record))
            result[locus][gene_id] = protein_ids
            print(f"  {gene_id}: {len(protein_ids)} proteins")

            # NCBI's request limit (10/sec with an API key)
            time.sleep(0.1)

        except Exception as expt:
            print(f"  Error with {locus} / {gene_id}: {expt}")
            result[locus][gene_id] = []

ABCA12: 35 -> ['26154', '694427', '459924', '100398127']
  26154: 4 proteins
  694427: 1 proteins
  459924: 3 proteins
  100398127: 1 proteins
  138382836: 1 proteins
  129488387: 1 proteins
  129031917: 1 proteins
  128590482: 1 proteins
  126932454: 1 proteins
  123644041: 1 proteins
  117094567: 1 proteins
  116810341: 1 proteins
  116551837: 1 proteins
  112636208: 2 proteins
  111546013: 1 proteins
  108543177: 1 proteins
  108308924: 1 proteins
  105867035: 1 proteins
  105816135: 1 proteins
  105712674: 1 proteins
  105580153: 1 proteins
  105530368: 1 proteins
  105504789: 1 proteins
  105481164: 1 proteins
  104669237: 1 proteins
  103255916: 1 proteins
  103217792: 1 proteins
  102118220: 1 proteins
  101137800: 1 proteins
  101047027: 1 proteins
  101003320: 1 proteins
  100971773: 1 proteins
  100956879: 1 proteins
  100593810: 1 proteins
  100448992: 1 proteins
ACAD9: 35 -> ['28976', '704527', '460679', '101865244']
  28976: 4 proteins
  704527: 5 proteins
  460679: 3 prot

* function: extract all protein ids per orthologs per genes into a list 

In [51]:
# create fasta folder
!mkdir fasta_lociii

def flatten_protein_ids(variant_dict):
    """
    Convert:
        {"variant1": [ids], "variant2": [ids]}
    into one unique list of protein IDs for that gene.
    """
    seq_ids = []
    seen = set()

    for variant_id, protein_ids in variant_dict.items():
        for pid in protein_ids:
            if pid not in seen:
                seq_ids.append(pid)
                seen.add(pid)

    return seq_ids



9339.80s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


* function: fetch protein details ('records') from NCBI
* modified

In [52]:
def fetch_protein_records(seq_ids):
    """
    Fetch GenBank protein records from NCBI.
    Uses chunks (at most 200) so the request does not become too large.
    """
    records = []
    chunk_size= 200
    for start in range(0, len(seq_ids), chunk_size):
        chunk = seq_ids[start:start + chunk_size]

        handle= Entrez.efetch(db="protein", rettype="gb", retmode="text", id=",".join(chunk))
        records.extend(list(SeqIO.parse(handle, "gb")))
        handle.close()

        time.sleep(0.35)

    return records

* function: get specie name from record eg 'Homo sapiens'

In [45]:
def get_species(record):
    """
    Retrieve species name from NCBI sequence record.

    Parameters
    ----------
    record : Bio.SeqRecord
        Sequence record.

    Returns
    -------
    str
        Species name.
    """
    description = record.description
    species = description.split(" [")[1][:-1]
    return species

* function; collect unique sequences; Homo sapiens first, then non Homo sapiens

In [53]:
def collect_unique_sequences_human_first(records):
    """
    Keep unique protein sequences.
    First collect unique Homo sapiens sequences,
    then collect unique non-human sequences.
    """
    seq_to_index = {}
    unique_records = []
    excluded_records = []

    inc = 0
    exc = 0

    # 1. Collect unique human sequences first
    for seq_record in records:
        sp = get_species(seq_record)
        if sp == "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}")
                inc += 1

    # 2. Collect other unique non-human sequences
    for seq_record in records:
        sp = get_species(seq_record)

        if sp != "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    return unique_records, excluded_records, inc, exc


* function; make a dict ('sequence_dict') comprising protein and its sequence

In [54]:
def make_sequence_dict(variant_dict, records):
    """
    Preserve your original nested structure, but replace each protein ID
    with its actual protein sequence.
    """
    #seq_by_id = {seq_record.id: str(seq_record.seq) for seq_record in records}
    seq_by_id = {}
    for seq_record in records:
        seq_by_id[seq_record.id: str(seq_record.seq)]= {}

    sequence_dict = {}

    for variant_id, protein_ids in variant_dict.items():
        sequence_dict[variant_id] = {}

        for pid in protein_ids:
            sequence_dict[variant_id][pid] = seq_by_id.get(pid, None)

    return sequence_dict

* RUN CODE USING CREATED FUNCTIONS
* Store dictionary containing protein id, unique sequences, etc 'summary_df' as dataframe
* save sequences in fasta file per gene

In [56]:
result.items()
#print(datetime.now())
import os
os.getcwd()
os.listdir()

['find_CPDs_Mary.ipynb',
 'clinvar',
 'clean_up_Mary.ipynb',
 'kondrashov.py',
 '.DS_Store',
 'orthologs_xml_updated.ipynb',
 'Tutorial_Pandas.ipynb',
 'fasta',
 'pathogenic',
 'fasta_lociii',
 'README.md',
 'orthologs_Rodentia.ipynb',
 'aln_match',
 'Validate_CPDs_Mary.ipynb',
 '.gitignore',
 'fasta_match_Copy',
 'fasta_match',
 'find_closest.ipynb',
 '.github',
 'find_CPDs_Match.ipynb',
 'orthologs.ipynb',
 'orthologs_xml.ipynb',
 '.ipynb_checkpoints',
 'find_CPDs.ipynb',
 '.git',
 'orthologs_Mary_Primates.ipynb',
 'find_closest_Mary.ipynb',
 'Outputs',
 'clean_up.ipynb',
 'aln_match - Copy',
 'align.ipynb',
 'pathogen_Mary.ipynb',
 'orthologs_new.ipynb',
 'pathogen.ipynb',
 'aln',
 'clinvar1']

In [57]:
protein_sequences = {}
unique_records_by_gene = {}
excluded_records_by_gene = {}

summary = []

for gene, variant_dict in result.items():

    print("\n" + "=" * 80) #demarcation line
    print(datetime.now())
    print(f"{gene} orthologs")

    # Get all protein IDs for this gene from your result dictionary
    seq_ids = flatten_protein_ids(variant_dict)

    print(f"{gene}: {len(seq_ids)} protein IDs")
    print(f"Sequence IDs: {seq_ids[:10]}{' ...' if len(seq_ids) > 10 else ''}")

    if len(seq_ids) == 0:
        protein_sequences[gene] = {}
        unique_records_by_gene[gene] = []
        excluded_records_by_gene[gene] = []

        summary.append({
            "gene": gene,
            "n_protein_ids": 0,
            "n_records_fetched": 0,
            "n_unique_sequences": 0,
            "n_excluded_duplicates": 0,
            "fasta_file": None
        })

        continue

    # Fetch protein records from NCBI
    records = fetch_protein_records(seq_ids)

    print(f"{gene}: {len(records)} protein records fetched")

    # Store nested dictionary of actual sequences
    protein_sequences[gene] = make_sequence_dict(variant_dict, records)

    # Collect unique sequences, human first
    print(f"\n{gene} orthologs: unique primate sequences\n")

    unique_records, excluded_records, inc, exc = collect_unique_sequences_human_first(records)

    unique_records_by_gene[gene] = unique_records
    excluded_records_by_gene[gene] = excluded_records

    # Save FASTA file for that gene
    fasta_path = f"fasta_lociii/{gene}.fasta"

    with open(fasta_path, "w") as output:
        SeqIO.write(unique_records, output, "fasta")

    print(f"\nTotal: {inc} unique sequences, {exc} excluded")
    print(f"{fasta_path} saved!")

    summary.append({
        "gene": gene,
        "n_protein_ids": len(seq_ids),
        "n_records_fetched": len(records),
        "n_unique_sequences": inc,
        "n_excluded_duplicates": exc,
        "fasta_file": fasta_path
    })

summary_df = pd.DataFrame(summary)
summary_df


2026-08-28 12:50:21.451992
ABCA12 orthologs
ABCA12: 41 protein IDs
Sequence IDs: ['NP_056472.2', 'NP_775099.2', 'XP_011509253.1', 'XP_054197317.1', 'XP_028686914.2', 'XP_016805928.3', 'XP_063646804.1', 'XP_063646805.1', 'XP_035161505.3', 'XP_069323576.1'] ...
ABCA12: 41 protein records fetched

ABCA12 orthologs: unique primate sequences

0:	NP_056472.2	(2277 aa)	glucosylceramide transporter ABCA12 isoform b [Homo sapiens]
1:	NP_775099.2	(2595 aa)	glucosylceramide transporter ABCA12 isoform a [Homo sapiens]
2:	XP_011509253.1	(2598 aa)	glucosylceramide transporter ABCA12 isoform X1 [Homo sapiens]
3:	XP_054197317.1	(2598 aa)	glucosylceramide transporter ABCA12 isoform X1 [Homo sapiens]
4:	XP_028686914.2	(2595 aa)	glucosylceramide transporter ABCA12 [Macaca mulatta]
5:	XP_016805928.3	(2595 aa)	glucosylceramide transporter ABCA12 isoform X3 [Pan troglodytes]
6:	XP_063646804.1	(2141 aa)	glucosylceramide transporter ABCA12 isoform X1 [Pan troglodytes]
7:	XP_063646805.1	(2138 aa)	glucosylcera

,gene,n_protein_ids,n_records_fetched,n_unique_sequences,n_excluded_duplicates,fasta_file
0,ABCA12,41,41,41,0,fasta_lociii/ABCA12.fasta
1,ACAD9,104,104,80,24,fasta_lociii/ACAD9.fasta
2,ACADM,111,111,91,20,fasta_lociii/ACADM.fasta
3,ACADVL,101,101,92,9,fasta_lociii/ACADVL.fasta
4,ACAT1,107,107,64,43,fasta_lociii/ACAT1.fasta
...,...,...,...,...,...,...
146,WRN,258,258,126,132,fasta_lociii/WRN.fasta
147,WT1,205,205,123,82,fasta_lociii/WT1.fasta
148,XPC,105,105,98,7,fasta_lociii/XPC.fasta
149,ZEB2,202,202,105,97,fasta_lociii/ZEB2.fasta
